## Data Quality and Cleaning
**Data Engineer**

In [58]:
import json
import pandas as pd
from pathlib import Path
import numpy as np

In [59]:
# Load the raw JSON
data_path = Path("../data/raw_credit_applications.json")  # or the correct relative/path in your repo
with open(data_path, "r") as f:
    raw_data = json.load(f)

df = pd.json_normalize(raw_data)

### Validity

In [60]:
# ensure numeric columns are numeric
numeric_cols = [
    "financials.annual_income",
    "financials.credit_history_months",
    "financials.debt_to_income",
    "financials.savings_balance",
    "decision.interest_rate",
    "decision.approved_amount",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [61]:
def check_numerical_for_limits(df, variable, lb, ub=None):
    """Check bounds [lb, ub) and replace out-of-bounds with NaN"""
    out_of_bounds = (df[variable] < lb)
    if ub is not None:
        out_of_bounds |= (df[variable] > ub)
    count_invalid = out_of_bounds.sum()
    pct_invalid = (count_invalid / len(df)) * 100
    print(f"{variable} out-of-bounds [{lb}, {ub or 'inf'}): {count_invalid} ({pct_invalid:.1f}%)")
    df.loc[out_of_bounds, variable] = np.nan
    return df

# flag and check numerical variables for upper and lower bounds
df = check_numerical_for_limits(df, "financials.annual_income", 0)
df = check_numerical_for_limits(df, "financials.savings_balance", 0)
df = check_numerical_for_limits(df, "financials.debt_to_income", 0, 1)
df = check_numerical_for_limits(df, "financials.credit_history_months", 0)
df = check_numerical_for_limits(df, "decision.approved_amount", 0)
df = check_numerical_for_limits(df, "decision.interest_rate", 0)

financials.annual_income out-of-bounds [0, inf): 0 (0.0%)
financials.savings_balance out-of-bounds [0, inf): 1 (0.2%)
financials.debt_to_income out-of-bounds [0, 1): 1 (0.2%)
financials.credit_history_months out-of-bounds [0, inf): 2 (0.4%)
decision.approved_amount out-of-bounds [0, inf): 0 (0.0%)
decision.interest_rate out-of-bounds [0, inf): 0 (0.0%)


In [62]:
# handle Datime consistency and transforming to age
today = pd.to_datetime('2026-03-05')
s = (
    df["applicant_info.date_of_birth"]
    .astype("string").str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "null": pd.NA})
)

d1 = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")
d2 = pd.to_datetime(s, errors="coerce", format="%Y/%m/%d")
d3 = pd.to_datetime(s, errors="coerce", format="%d/%m/%Y")
d4 = pd.to_datetime(s, errors="coerce", format="%m/%d/%Y")

m = s.str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
p1 = pd.to_numeric(s.str.split("-", expand=True)[0], errors="coerce")
d5 = pd.to_datetime(s.where(m), errors="coerce", format="%d-%m-%Y")
d6 = pd.to_datetime(s.where(m), errors="coerce", format="%m-%d-%Y")

df["applicant_info.date_of_birth"] = (d1.fillna(d2).fillna(d3).fillna(d4).fillna(d5.where(p1 > 12, d6)))

df['applicant_info.age'] = (today - df['applicant_info.date_of_birth']).dt.days // 365

In [63]:
# unify email adresses and check for consistency
df['applicant_info.email'] = df['applicant_info.email'].str.lower().str.strip()
invalid_email = ~df['applicant_info.email'].str.contains(r'@[^@]+\.[^@]+$', na=False)
print(f"Invalid emails (no @domain.tld): {invalid_email.sum()} ({invalid_email.mean()*100:.1f}%)")
df.loc[invalid_email, 'applicant_info.email'] = np.nan

# check names for consistency
single_word_names = df['applicant_info.full_name'].str.split().str.len() == 1
no_capital = ~df['applicant_info.full_name'].str[0].str.isupper()
print(f"Single-word names: {single_word_names.sum()} ({single_word_names.mean()*100:.1f}%)")
print(f"No capital start: {no_capital.sum()} ({no_capital.mean()*100:.1f}%)")

# transform ssn patterns
ssn_pattern = r'^\d{3}-\d{2}-\d{4}$'
invalid_ssn = ~df['applicant_info.ssn'].str.match(ssn_pattern, na=False)
print(f"Invalid SSN format (not XXX-XX-XXXX): {invalid_ssn.sum()} ({invalid_ssn.mean()*100:.1f}%)")
df.loc[invalid_ssn, 'applicant_info.ssn'] = np.nan

# ZIP: Enforce 5 digits (truncate extra)
zip_str = df['applicant_info.zip_code'].astype(str)
long_zip = zip_str.str.len() > 5
print(f"ZIP >5 digits: {long_zip.sum()}")
df.loc[long_zip, 'applicant_info.zip_code'] = zip_str.loc[long_zip].str[:5]
df['applicant_info.zip_code'] = pd.to_numeric(df['applicant_info.zip_code'], errors='coerce').astype('Int64')

# check for non-boolean loan_approved
print("Non-boolean count loan_approved:", (~df['decision.loan_approved'].isin([True, False])).sum())

Invalid emails (no @domain.tld): 10 (2.0%)
Single-word names: 0 (0.0%)
No capital start: 0 (0.0%)
Invalid SSN format (not XXX-XX-XXXX): 5 (1.0%)
ZIP >5 digits: 0
Non-boolean count loan_approved: 0


### Consistency

In [64]:
# transform gender for consistency
df['applicant_info.gender'] = (
    df['applicant_info.gender']
    .str.strip()
    .str.upper()
    .map({'M': 'Male', 'MALE': 'Male', 'F': 'Female', 'FEMALE': 'Female', '': np.nan})
)

In [65]:
# transform spending behaviour into dataframe
sp = df[["_id", "spending_behavior"]].explode("spending_behavior").copy()

sp_norm = pd.json_normalize(sp["spending_behavior"])
sp_norm.index = sp.index
sp = pd.concat([sp[["_id"]], sp_norm], axis=1)

sp["amount"] = pd.to_numeric(sp["amount"], errors="coerce")
sp_pivot = (sp.pivot_table(index="_id", columns="category", values="amount", aggfunc="sum", fill_value=0).add_prefix("spend_").reset_index())

df = df.drop(columns=["spending_behavior"]).merge(sp_pivot, on="_id", how="left")

In [66]:
# Check for duplicate application IDs
total_rows = df.shape[0]
unique_ids = df["_id"].nunique()
duplicate_rows = total_rows - unique_ids

print("Total rows:", total_rows)
print("Duplicate rows (by _id):", duplicate_rows)

before = df.shape[0]
df = df.drop_duplicates(subset="_id", keep="first")
after = df.shape[0]

df = df.copy(deep=True)
df['_temp_nan_count'] = df.isna().sum(axis=1)
idx = df.groupby('_id')['_temp_nan_count'].idxmin()
df = df.loc[idx].drop(columns='_temp_nan_count')

after = df.shape[0]
print("Rows before dropping duplicates:", before)
print("Rows after:", after)

Total rows: 502
Duplicate rows (by _id): 2
Rows before dropping duplicates: 502
Rows after: 500


In [67]:
# detect duplicate SSNs
df["duplicate_ssn"] = (df["applicant_info.ssn"].notna() & df.duplicated(subset="applicant_info.ssn", keep=False))

duplicate_ssn_rows = df[df["duplicate_ssn"]]
print(f"\nDuplicate SSNs found: {duplicate_ssn_rows['applicant_info.ssn'].nunique()}")
print(duplicate_ssn_rows[["_id", "applicant_info.ssn"]].sort_values("applicant_info.ssn"))

df_flag = df[df["duplicate_ssn"]].copy()
df = df[~df["duplicate_ssn"]].copy()

df = df.drop(columns=["duplicate_ssn"])
df_flag = df_flag.drop(columns=["duplicate_ssn"])


Duplicate SSNs found: 2
         _id applicant_info.ssn
122  app_016        780-24-9300
92   app_088        780-24-9300
16   app_101        937-72-8731
499  app_234        937-72-8731


### Completeness

In [68]:
# combine salary and income into one column
# done prior to drop salary in next step before cleaning data
print("Pre-merge: income NaN:", df['financials.annual_income'].isnull().sum())
df['financials.annual_income'] = df['financials.annual_income'].fillna(df['financials.annual_salary'])
print("Post-merge income NaN:", df['financials.annual_income'].isnull().sum())

Pre-merge: income NaN: 5
Post-merge income NaN: 0


In [69]:
# 1. Overview of missing data
def missing_summary(df):
    missing_counts = df.isna().sum().sort_values(ascending=False)
    missing_percent = (missing_counts / len(df) * 100).round(1)
    return pd.DataFrame({
        "missing_count": missing_counts,
        "missing_percent": missing_percent
    })


ms = missing_summary(df)
print(ms)

                                  missing_count  missing_percent
notes                                       496            100.0
financials.annual_salary                    491             99.0
loan_purpose                                447             90.1
processing_timestamp                        434             87.5
decision.rejection_reason                   292             58.9
decision.approved_amount                    204             41.1
decision.interest_rate                      204             41.1
applicant_info.email                         10              2.0
applicant_info.age                            4              0.8
applicant_info.date_of_birth                  4              0.8
applicant_info.ip_address                     4              0.8
applicant_info.ssn                            4              0.8
applicant_info.gender                         2              0.4
financials.credit_history_months              2              0.4
financials.debt_to_income

In [70]:
"""
# Drop columns with >50% missing values
threshold = 50
cols_to_drop = ms[ms["missing_percent"] > threshold].index.tolist()
print("Dropping columns with >50% missing:", cols_to_drop)

df = df.drop(columns=cols_to_drop)
print("\nRemaining columns:", df.shape[1])
"""
df = df.drop(columns=["financials.annual_salary", "notes"])

In [71]:
path = Path("../data/clean_credit_applications.csv")
df.to_csv(path, index=False)
print("Shape:", df.shape)
df

Shape: (496, 34)


,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,...,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
383,app_001,NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,102000.0,...,1152,0,0,0,0,0,0,0,0,0
339,app_002,NaN,Kevin Roberts,kevin.roberts9@protonmail.com,992-61-4010,172.19.95.144,Male,1999-08-01,10020,41000.0,...,0,0,0,0,0,0,0,0,0,0
284,app_003,NaN,Lisa Gonzalez,lisa.gonzalez51@yahoo.com,833-33-5929,172.21.35.195,Female,1982-08-24,90213,65000.0,...,0,0,0,450,0,0,0,0,0,0
255,app_004,NaN,Karen Nelson,karen.nelson35@outlook.com,486-50-5539,172.31.79.76,Female,1995-02-28,90217,69000.0,...,0,0,810,0,0,0,0,329,0,0
136,app_005,NaN,Christine Mitchell,christine.mitchell3@outlook.com,400-91-8156,172.25.44.173,Female,1960-06-19,90296,39000.0,...,0,0,0,0,585,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269,app_496,NaN,Rachel Miller,rachel.miller69@protonmail.com,953-69-6408,10.194.95.93,Female,1979-11-12,90211,87000.0,...,175,0,0,0,0,0,0,0,0,0
45,app_497,NaN,Patrick Wilson,patrick.wilson77@hotmail.com,655-58-3025,10.131.43.18,Male,1994-04-20,90233,48000.0,...,0,0,423,0,0,0,0,0,0,0
191,app_498,NaN,James Rivera,james.rivera25@gmail.com,942-34-6834,172.18.221.237,Male,1989-10-19,90276,86000.0,...,93,0,0,0,0,0,0,0,0,0
302,app_499,NaN,Deborah Lee,deborah.lee74@mail.com,843-60-2218,10.1.111.83,Female,1983-12-02,90218,111000.0,...,0,0,810,0,0,0,0,0,0,0
